# Day 1 — LLM Routing, Model Behaviour and Agent Design Foundations

### Learning outcomes
By the end of this practical you should be able to compare hosted and local inference, observe provider/model/latency metadata, understand why an LLM call is not yet an agent, preserve conversational state explicitly, and create a bounded agent specification for a realistic fulfilment-centre support scenario.

>**Dr Julius Sechang Mboli**
>
>**DAIM, University of Hull**
>
>**https://www.hull.ac.uk/staff-directory/julius-mboli**
>
>**https://www.linkedin.com/in/engr-julius-sechang-mboli/**
>
>**https://jsmboli.github.io/**

In [ ]:
from pathlib import Path
import os, sys, json, time, importlib.util
import pandas as pd
pd.set_option('display.max_colwidth', None)

# Locate the package without relying on a fixed working directory.
cwd = Path.cwd().resolve()
RESOURCE_DIR = None
for p in [cwd, *cwd.parents]:
    if (p / "resources" / "src").exists():
        RESOURCE_DIR = p / "resources"
        break
    if (p / "src").exists() and (p / "notebooks").exists() and (p / "data").exists():
        RESOURCE_DIR = p
        break
if RESOURCE_DIR is None:
    raise RuntimeError("Could not locate resources/src. Extract the complete bootcamp ZIP and open this notebook from inside it.")

SRC = RESOURCE_DIR / "src"
DATA = RESOURCE_DIR / "data"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from providers import ProviderRouter, ProviderError, make_messages
from notebook_utils import (
    environment_table, provider_table, response_table,
    run_with_progress, progress_indicator, display_response, display_note,
)

router = ProviderRouter(verbose=True)
display(environment_table(RESOURCE_DIR, router))

In [ ]:
status = run_with_progress("Checking Groq, Ollama and optional providers", router.diagnose, True)
display(provider_table(status))

PRIMARY_PROVIDER = (
    "groq" if status["groq"]["available"]
    else "ollama" if status["ollama"]["available"]
    else "mimic"
)
print("Primary live provider for this notebook:", PRIMARY_PROVIDER)
print("Groq model:", status["groq"].get("model"))
print("Ollama model:", status["ollama"].get("model"))

## 1. Same business task, different providers

A useful provider abstraction lets us keep the **task constant** while changing the inference backend. This makes differences in latency, output length and operational constraints visible without rewriting the application.

We will compare every provider that is genuinely available; the offline mimic route remains a control condition rather than pretending to be a real LLM.

In [3]:
from tqdm.auto import tqdm

prompt = """
You are advising an adult learner preparing for a UK fulfilment-centre industrial visit.
Propose four bounded agentic-AI use cases across receiving, stowing, picking, packing or exception handling.
For each use case give: user, goal, data/tool, state, permitted action, prohibited action, human-review point, and one failure mode.
Do not claim access to Amazon internal systems or non-public data.
"""

records=[]
responses={}
for provider in tqdm(["groq", "ollama", "mimic"], desc="Provider comparison"):
    if provider != "mimic" and not status[provider]["available"]:
        continue
    try:
        r = run_with_progress(
            f"{provider} — bounded use-case analysis",
            router.chat, make_messages(prompt), provider=provider,
            max_tokens=650, fallback_on_error=False,
        )
        responses[provider]=r
        records.append(r.summary())
    except Exception as exc:
        records.append({"provider":provider,"model":None,"latency_s":None,"error":str(exc)})

display(pd.DataFrame(records))
for provider,r in responses.items():
    display_response(r, f"{provider.upper()} answer")

Provider comparison:   0%|          | 0/3 [00:00<?, ?it/s]

▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=650
✓ Groq completed in 1.509s | model=qwen/qwen3.6-27b


▶ Ollama request started | model=qwen3.5:0.8b | think=False | max_tokens=650 | host=http://127.0.0.1:11434
✓ Ollama completed in 3.900s | model=qwen3.5:0.8b


,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,groq,qwen/qwen3.6-27b,1.509,120.0,650.0,770.0,length,chatcmpl-a03ea1d0-78de-436d-a5d2-1d064249d177,https://api.groq.com/openai/v1/chat/completions,format=hidden; effort=none
1,ollama,qwen3.5:0.8b,3.900,120.0,650.0,770.0,length,NaN,http://127.0.0.1:11434/api/chat,thinking disabled
2,mimic,deterministic-classroom-mimic,0.000,NaN,NaN,NaN,NaN,NaN,offline,none


### GROQ answer

...

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,ollama,qwen3.5:0.8b,3.9,120,650,770,length,None,http://127.0.0.1:11434/api/chat,thinking disabled


Here is a structured proposal for four bounded agentic-AI use cases tailored for an adult learner preparing for a UK fulfilment-centre industrial visit. These scenarios focus on real-world logistics workflows, adhere strictly to safety guidelines regarding proprietary/internal system access (as requested), and are designed as educational exercises rather than operational directives.

### 1. Receiving & Inventory Verification
**User:** Logistics Coordinator / Warehouse Supervisor
*   **Goal:** Verify incoming goods against a digital inventory ledger before unloading or moving them, ensuring no stock discrepancies exist between the physical warehouse floor and the system database.
*   **Data/Tool:** Digital ERP/WMS System (e.g., SAP, Oracle), Barcode Scanner Interface, Physical Inventory Count Sheet.
*   **State:** The agent is currently at a receiving dock; it has received an incoming shipment but needs to cross-reference this receipt with its local inventory count and the global ledger status before unloading or moving goods.
*   **Permitted Action:** Scan barcodes of all items in the inbound bin, compare each item's barcode against the current system record ID/quantity, flag any discrepancies (e.g., "Item X has 10 units on receipt but only 5 remain"), and generate a status report for approval before unloading.
*   **Prohibited Action:** Attempt to modify inventory records directly from this interface without explicit user confirmation or bypassing the system's integrity checks; do not attempt to delete goods that are already marked as "in transit" in the ledger.
*   **Human-Review Point:** Ensure the discrepancy flag is accurate before proceeding with unloading, especially if there is a known historical error (e.g., an old batch of stock).

### 2. Stowing & Loading Optimization
**User:** Fleet Manager / Logistics Planner
*   **Goal:** Optimize the loading process to minimize empty space and maximize cargo volume per truck while adhering to strict weight limits for specific vehicle types or environmental regulations in a UK setting (e.g., avoiding overloading).
*   **Data/Tool:** Vehicle Load Calculator, GPS Tracking System, Weight Limits Database, Truck Routing Software.
*   **State:** The agent is at the loading dock; it has received multiple truck loads and needs to calculate optimal stowage heights based on vehicle dimensions (e.g., Mercedes V-Class) and weight limits to ensure safe stacking without compromising structural integrity or hitting cargo bay walls.
*   **Permitted Action:** Analyze current load configurations, apply height/weight constraints from the database, recommend specific pallet layouts for maximum space utilization while staying within safety zones, and generate a loading plan with estimated travel times based on route optimization algorithms.
*   **Prohibited Action:** Attempt to calculate optimal stowage without considering vehicle dimensions or weight limits; do not suggest stacking methods that violate UK road regulations (e.g., placing heavy items at the very bottom of a truck bed).

### 3. Picking & Sorting with Exception Handling
**User:** Warehouse Operations Manager / Quality Control Lead
*   **Goal:** Automate picking and sorting tasks while detecting anomalies in real-time to prevent stockouts or quality issues, ensuring that only

### MIMIC answer

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,mimic,deterministic-classroom-mimic,0.0,None,None,None,None,None,offline,none


A bounded fulfilment-centre agent can retrieve approved process guidance, summarise exceptions, calculate operational metrics and pause sensitive actions for human review.

### Discussion prompts
1. Which provider gave the fastest first useful answer?
2. Which answer was most grounded in the constraints we explicitly provided?
3. Does a larger/hosted model automatically make the workflow safer? Why not?
4. What data must stay local in a real organisation, and what could be sent to a hosted API?
5. Which measurements would you record in production: latency, tokens, errors, tool calls, human overrides, cost, groundedness?

## 2. Application-managed conversational state

Most chat APIs do not remember a previous call unless the application resends prior messages or an orchestration layer persists them. Here we make the state visible in a Python list before introducing LangGraph state tomorrow.

In [4]:
conversation = [
    {"role":"system","content":"You are a precise operations-analytics teaching assistant. Never invent access to company systems."},
    {"role":"user","content":"We are designing a packing-exception support agent. Define a safe scope in five bullets."},
]

r1 = run_with_progress(
    "Conversation turn 1", router.chat, conversation,
    provider=PRIMARY_PROVIDER, max_tokens=350,
    fallback_on_error=(PRIMARY_PROVIDER=="mimic")
)
display_response(r1, "Turn 1")
conversation += [{"role":"assistant","content":r1.text}, {
    "role":"user","content":"Now add three tools, state fields, an audit log and one human approval gate without expanding the scope."
}]
r2 = run_with_progress(
    "Conversation turn 2", router.chat, conversation,
    provider=PRIMARY_PROVIDER, max_tokens=450,
    fallback_on_error=(PRIMARY_PROVIDER=="mimic")
)
display_response(r2, "Turn 2")
print("Messages currently held by the application:", len(conversation)+1)

▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=350
✓ Groq completed in 0.837s | model=qwen/qwen3.6-27b


### Turn 1

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,groq,qwen/qwen3.6-27b,0.837,51,323,374,stop,chatcmpl-bb32f343-8cbb-4969-8124-043c3d299602,https://api.groq.com/openai/v1/chat/completions,format=hidden; effort=none


Here is a safe scope for a packing-exception support agent, designed to minimize liability and operational risk:

*   **Information Retrieval Only**: The agent is restricted to reading and summarizing existing data from the Order Management System (OMS) and Warehouse Management System (WMS), such as packing slip contents, weight discrepancies, and photos of packed items, without the ability to modify records or trigger financial transactions.
*   **Standardized Resolution Pathways**: The agent can only recommend pre-approved, low-risk actions defined by operations policy (e.g., "initiate replacement shipment," "schedule pickup for missing item," or "issue partial refund up to $X"), requiring human supervisor approval for any exception outside these predefined parameters.
*   **No Liability Admission or Legal Interpretation**: The agent is strictly prohibited from admitting fault, assigning blame to specific employees or carriers, or providing legal advice regarding consumer protection laws; all responses must be neutral and factual, directing complex liability disputes to human legal or senior support teams.
*   **Data Privacy and Access Control**: The agent operates within strict role-based access controls (RBAC), ensuring it only accesses customer data relevant to the specific ticket at hand and never exposes sensitive PII (Personally Identifiable Information) or payment details in chat logs or external integrations.
*   **Human-in-the-Loop Escalation**: The agent must automatically escalate any case involving high-value items, repeated exceptions for the same customer, ambiguous evidence (e.g., unclear photos), or customer requests for compensation beyond standard policy limits to a human agent for manual review.

▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=450
✓ Groq completed in 1.207s | model=qwen/qwen3.6-27b


### Turn 2

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,groq,qwen/qwen3.6-27b,1.207,404,450,854,length,chatcmpl-c3df842f-359d-4878-971e-df4df2ca3997,https://api.groq.com/openai/v1/chat/completions,format=hidden; effort=none


Here are the specific technical components added to the packing-exception support agent, strictly adhering to the previously defined safe scope:

**Three Tools**
1.  **`get_packing_manifest`**: A read-only API call to the WMS that retrieves the expected item list, weights, and associated packing slip images for a specific Order ID.
2.  **`check_customer_history`**: A read-only query to the CRM/OMS that returns the count of previous packing exceptions for the customer in the last 90 days (used to trigger escalation if >2).
3.  **`generate_resolution_draft`**: A template engine that creates a pre-approved response draft (e.g., "Replacement Shipment Initiated") based on the exception type, but does *not* execute the action; it only outputs the text and required approval metadata.

**State Fields**
1.  `exception_type`: Enum (e.g., `MISSING_ITEM`, `WRONG_ITEM`, `DAMAGED_IN_PACK`) determined by initial classification.
2.  `escalation_flag`: Boolean, set to `true` if `check_customer_history` returns >2 prior exceptions or if the item value exceeds the predefined threshold.
3.  `approval_status`: Enum (`PENDING_HUMAN_REVIEW`, `APPROVED`, `REJECTED`), initialized to `PENDING_HUMAN_REVIEW` for all non-trivial resolutions.

**Audit Log**
*   **Structure**: Immutable log entry created for every tool call and state change.
*   **Fields**: `timestamp`, `agent_id`, `order_id`, `action_performed` (e.g., "Called get_packing_manifest"), `input_parameters` (sanitized), `output_summary` (e.g., "Retrieved 3 items"), and `user_session_id`.
*   **Purpose**: Provides a complete, tamper-evident trail for compliance reviews and debugging, ensuring no action is taken without a recorded justification.

**One Human Approval Gate**
*   **Trigger**: Activated when `approval_status` is `PENDING_HUMAN_REVIEW` and the agent attempts to finalize a resolution

Messages currently held by the application: 5


## 3. Structured agent card

An agent design should be explicit enough to review before implementation. The following schema separates **goal**, **tools**, **state**, **authority**, **human oversight** and **failure handling**.

In [5]:
structured_prompt = """
Return JSON only with this schema:
{
  "agent_name": string,
  "primary_user": string,
  "goal": string,
  "inputs": [string],
  "tools": [{"name": string, "purpose": string}],
  "state_fields": [string],
  "allowed_actions": [string],
  "prohibited_actions": [string],
  "human_review_triggers": [string],
  "audit_events": [string],
  "failure_modes": [string]
}
Design a bounded packing-exception support agent for a teaching simulation.
"""
r = run_with_progress(
    "Generating structured agent card", router.chat, make_messages(structured_prompt),
    provider=PRIMARY_PROVIDER, max_tokens=650, fallback_on_error=(PRIMARY_PROVIDER=="mimic")
)
display_response(r, "Structured agent-card generation")

# Parse only when the provider really returned JSON.
try:
    card=json.loads(r.text)
    display(pd.json_normalize(card, sep="."))
except Exception as exc:
    print("JSON validation result: response was not strict JSON ->", exc)
    print("This is a useful reason to use structured-output features or schema validation in production.")

▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=650
✓ Groq completed in 1.456s | model=qwen/qwen3.6-27b


### Structured agent-card generation

,provider,model,latency_s,input_tokens,output_tokens,total_tokens,finish_reason,request_id,endpoint,reasoning_mode
0,groq,qwen/qwen3.6-27b,1.456,162,567,729,stop,chatcmpl-23882412-2e22-41c7-b9ec-357fb39e3ed9,https://api.groq.com/openai/v1/chat/completions,format=hidden; effort=none


{
  "agent_name": "PackBot Support Assistant",
  "primary_user": "E-commerce Customer Service Representative",
  "goal": "Assist human agents in resolving customer inquiries regarding damaged, missing, or incorrect items in shipments by retrieving order details and generating standardized resolution templates.",
  "inputs": [
    "Customer Order ID",
    "Customer Reported Issue Description",
    "Customer Contact Information"
  ],
  "tools": [
    {
      "name": "OrderLookupService",
      "purpose": "Retrieve order details, shipping status, and itemized contents from the database."
    },
    {
      "name": "InventoryCheckService",
      "purpose": "Verify current stock levels for replacement items."
    },
    {
      "name": "ResolutionTemplateGenerator",
      "purpose": "Draft standardized email responses for refunds, replacements, or partial credits based on policy rules."
    },
    {
      "name": "CaseLogWriter",
      "purpose": "Record interaction details and resolution steps into the CRM system."
    }
  ],
  "state_fields": [
    "current_order_id",
    "issue_category",
    "resolution_status",
    "customer_sentiment_score",
    "last_action_taken"
  ],
  "allowed_actions": [
    "Query order details",
    "Check inventory availability",
    "Generate draft response",
    "Log case notes",
    "Escalate to human supervisor"
  ],
  "prohibited_actions": [
    "Process financial transactions directly",
    "Modify customer account permissions",
    "Send final communications without human approval",
    "Access PII beyond what is necessary for the specific order",
    "Alter historical order records"
  ],
  "human_review_triggers": [
    "Refund amount exceeds $50",
    "Customer sentiment score indicates high frustration",
    "Inventory check shows item is out of stock",
    "Issue category is classified as 'Fraud Suspected'",
    "Agent fails to resolve issue within 3 interaction turns"
  ],
  "audit_events": [
    "Order data accessed",
    "Resolution template generated",
    "Escalation initiated",
    "Case notes logged",
    "Human override applied"
  ],
  "failure_modes": [
    "Order ID not found in database",
    "Inventory service timeout",
    "Ambiguous issue description requiring clarification",
    "Policy rule conflict for resolution type",
    "CRM write failure"
  ]
}

,agent_name,primary_user,goal,inputs,tools,state_fields,allowed_actions,prohibited_actions,human_review_triggers,audit_events,failure_modes
0,PackBot Support Assistant,E-commerce Customer Service Representative,Assist human agents in resolving customer inqu...,"[Customer Order ID, Customer Reported Issue De...","[{'name': 'OrderLookupService', 'purpose': 'Re...","[current_order_id, issue_category, resolution_...","[Query order details, Check inventory availabi...","[Process financial transactions directly, Modi...","[Refund amount exceeds $50, Customer sentiment...","[Order data accessed, Resolution template gene...","[Order ID not found in database, Inventory ser..."


## 4. From LLM to agent

An **LLM call** maps messages to a response. An **agentic system** adds state, tools, observations, iteration, authority boundaries, persistence, and often human intervention. Tomorrow we implement those components with LangGraph rather than merely describing them in a prompt.

**Preparation:** sketch the state dictionary your agent would need after each step and identify which steps should be deterministic rather than decided by the LLM.